# Medical Guidelines: Fixed-Size Chunking Pipeline

## What does this notebook do?
We have 10 medical guideline PDFs. An AI system (such as a RAG chatbot) cannot read an entire PDF at once because these documents are very long. So we split the text into **small, equally sized pieces (chunks)**.

**Example:** Think of splitting a book into pages of 500 words each, so that when someone asks a question, we can quickly pull out only the relevant page instead of handing over the whole book.

## Pipeline steps
1. **Imports:** Load the libraries we need ← *we are here*
2. Extract text from the PDFs
3. Apply fixed-size chunking
4. Analyze and validate the chunks
5. Save the chunks

## Step 1: Imports
Here we load all the libraries used in this notebook. Each one has a small job:
- `pathlib`: finds files and folders
- `fitz` (PyMuPDF): reads text out of PDFs
- `re`: regular expressions, used for text cleaning
- `pandas`: shows results in clean tables
- `tqdm`: progress bar while processing 10 files
- `json`: saves chunks to disk

In [1]:
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Standard library
import json
import re
from pathlib import Path

# Third-party libraries
import pymupdf  # PyMuPDF, used to extract text from PDFs
import pandas as pd
from tqdm.auto import tqdm  # progress bar

# Quick check that everything loaded, and which versions we have
print("PyMuPDF version:", pymupdf.__version__)
print("pandas version:", pd.__version__)
print("All imports successful")

PyMuPDF version: 1.28.2
pandas version: 2.3.3
All imports successful


## Step 2: Locate the PDFs
Before extracting any text, we confirm that the notebook can see our `Data` folder and count the PDFs inside it. This catches path problems early, before we spend time processing 10 files.

What we check:
- The `Data` folder exists
- How many PDF files it contains (including subfolders)
- File names and sizes, so we can spot anything unusual (empty or very small files)

In [3]:
# Path to the folder that holds all guideline PDFs (next to this notebook)
DATA_DIR = Path("Data")

# Confirm the folder exists
print("Data folder exists:", DATA_DIR.exists())
print("Absolute path:", DATA_DIR.resolve())

# Find all PDFs, including subfolders and uppercase .PDF extensions
pdf_files = sorted(p for p in DATA_DIR.rglob("*") if p.suffix.lower() == ".pdf")
print(f"\nTotal PDFs found: {len(pdf_files)}")

# Build a small table with file name and size
files_df = pd.DataFrame(
    {
        "file_name": [p.name for p in pdf_files],
        "size_mb": [round(p.stat().st_size / (1024 * 1024), 2) for p in pdf_files],
    }
)

# Show the first 10 files, then the smallest and largest ones
print("\nFirst 10 files:")
display(files_df.head(10))

print("Smallest 5 files:")
display(files_df.nsmallest(5, "size_mb"))

print("Largest 5 files:")
display(files_df.nlargest(5, "size_mb"))

print(f"\nTotal size of all PDFs: {files_df['size_mb'].sum():.1f} MB")

Data folder exists: True
Absolute path: /home/as/Documents/week8/Med_Fixed_chunking/Data

Total PDFs found: 10

First 10 files:


,file_name,size_mb
0,Doc1.pdf,0.80
1,Doc10.pdf,1.21
2,Doc2.pdf,0.07
3,Doc3.pdf,3.38
4,Doc4.pdf,1.34
5,Doc5.pdf,1.97
6,Doc6.pdf,1.05
7,Doc7.pdf,6.74
8,Doc8.pdf,4.46
9,Doc9.pdf,1.75


Smallest 5 files:


,file_name,size_mb
2,Doc2.pdf,0.07
0,Doc1.pdf,0.80
6,Doc6.pdf,1.05
1,Doc10.pdf,1.21
4,Doc4.pdf,1.34


Largest 5 files:


,file_name,size_mb
7,Doc7.pdf,6.74
8,Doc8.pdf,4.46
3,Doc3.pdf,3.38
5,Doc5.pdf,1.97
9,Doc9.pdf,1.75



Total size of all PDFs: 22.8 MB


## Step 3: Extract Text (page by page)
Now we read the text out of every PDF using PyMuPDF. We keep each page separate and store:
- `doc_name`: which guideline the text came from
- `page_num`: the page number (starting at 1)
- `text`: the raw text of that page

**Why page by page?** Later, every chunk can point back to its exact source page, so answers from the AI can be verified against the original guideline.

**Important:** This step does *no cleaning*. We extract the text exactly as it appears, so we can inspect it and decide the cleaning rules together in the next step.

We also track:
- PDFs that fail to open
- Pages with no text (usually scanned images, which would need OCR)

In [4]:
pages = []      # one record per page
failed = []     # PDFs that could not be opened or read

for pdf_path in tqdm(pdf_files, desc="Extracting PDFs"):
    try:
        with pymupdf.open(pdf_path) as doc:
            for page_index, page in enumerate(doc):
                # sort=True orders text blocks top-to-bottom, left-to-right
                text = page.get_text("text", sort=True)
                pages.append(
                    {
                        "doc_name": pdf_path.name,
                        "page_num": page_index + 1,
                        "text": text,
                    }
                )
    except Exception as e:
        failed.append((pdf_path.name, str(e)))

pages_df = pd.DataFrame(pages)
pages_df["char_count"] = pages_df["text"].str.len()

print(f"PDFs processed: {len(pdf_files) - len(failed)} / {len(pdf_files)}")
print(f"Total pages extracted: {len(pages_df)}")
print(f"Total characters: {pages_df['char_count'].sum():,}")

# Failed PDFs
print(f"\nFailed PDFs: {len(failed)}")
for name, err in failed:
    print(f"  {name}: {err}")

# Pages with (almost) no text, likely scanned images
empty_pages = pages_df[pages_df["char_count"] < 20]
print(f"\nPages with almost no text: {len(empty_pages)}")

# Per-document summary: pages, characters, and empty pages
summary = (
    pages_df.groupby("doc_name")
    .agg(
        pages=("page_num", "count"),
        total_chars=("char_count", "sum"),
        empty_pages=("char_count", lambda s: (s < 20).sum()),
    )
    .sort_values("empty_pages", ascending=False)
)
print("\nDocuments with the most empty pages:")
display(summary.head(10))

# Peek at one real page so we can see what the raw text looks like
sample = pages_df[pages_df["char_count"] > 500].iloc[0]
print(f"\nSample raw text: {sample['doc_name']}, page {sample['page_num']}")
print("-" * 60)
print(sample["text"][:1200])

Extracting PDFs:   0%|          | 0/10 [00:00<?, ?it/s]

PDFs processed: 10 / 10
Total pages extracted: 1308
Total characters: 5,611,897

Failed PDFs: 0

Pages with almost no text: 18

Documents with the most empty pages:


,pages,total_chars,empty_pages
doc_name,,,
Doc6.pdf,184,694166,4
Doc7.pdf,124,204655,4
Doc10.pdf,90,292447,3
Doc1.pdf,61,204231,3
Doc3.pdf,494,2826831,1
Doc4.pdf,50,146167,1
Doc9.pdf,149,659759,1
Doc5.pdf,94,411409,1
Doc2.pdf,12,23063,0



Sample raw text: Doc1.pdf, page 4
------------------------------------------------------------
Guideline for the pharmacological treatment of hypertension in adults

ISBN 978-92-4-003398-6 (electronic version)
ISBN 978-92-4-003397-9 (print version)


© World Health Organization 2021

Some rights reserved. This work is available under the Creative Commons Attribution-NonCommercial-
ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO;
https://creativecommons.org/licenses/by-nc-sa/3.0/igo).

Under the terms of this licence, you may copy, redistribute and adapt the work for non-commercial
purposes, provided the work is appropriately cited, as indicated below. In any use of this work, there
should be no suggestion that WHO endorses any specific organization, products or services. The use of
the WHO logo is not permitted. If you adapt the work, then you must license your work under the same
or equivalent Creative Commons licence. If you create a translation of this work, you should add the
foll

## Step 4: Save Raw Text to .txt Files
We now save the extracted text to disk, **exactly as extracted, with no cleaning**.

- One `.txt` file per PDF, stored in the `raw_text` folder (e.g. `Doc1.pdf` becomes `Doc1.txt`)
- Each page starts with a marker like `[[PAGE 4]]`, so we can always trace text back to its source page
- Files are saved in UTF-8 encoding, so special medical symbols (e.g. µ, ≥, °) are preserved

**Why save raw text?** Extraction took about 40 seconds. By saving it, we never need to re-read the PDFs, and the chunking step can start directly from these files.

In [5]:
# Folder where the raw text files will be saved
RAW_TEXT_DIR = Path("raw_text")
RAW_TEXT_DIR.mkdir(exist_ok=True)

# Write one .txt file per PDF, with a page marker before each page
for doc_name, group in pages_df.groupby("doc_name"):
    out_path = RAW_TEXT_DIR / f"{Path(doc_name).stem}.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        for _, row in group.sort_values("page_num").iterrows():
            f.write(f"[[PAGE {row['page_num']}]]\n")
            f.write(row["text"])
            f.write("\n\n")

# Verify what was saved
saved = sorted(RAW_TEXT_DIR.glob("*.txt"))
print(f"Saved {len(saved)} files to: {RAW_TEXT_DIR.resolve()}\n")

check_df = pd.DataFrame(
    {
        "file": [p.name for p in saved],
        "size_mb": [round(p.stat().st_size / (1024 * 1024), 2) for p in saved],
    }
)
display(check_df)

# Show the start of one saved file to confirm the format
print("Preview of Doc2.txt:")
print("-" * 60)
print((RAW_TEXT_DIR / "Doc2.txt").read_text(encoding="utf-8")[:600])

Saved 10 files to: /home/as/Documents/week8/Med_Fixed_chunking/raw_text



,file,size_mb
0,Doc1.txt,0.20
1,Doc10.txt,0.28
2,Doc2.txt,0.02
3,Doc3.txt,2.71
4,Doc4.txt,0.14
5,Doc5.txt,0.40
6,Doc6.txt,0.67
7,Doc7.txt,0.20
8,Doc8.txt,0.14
9,Doc9.txt,0.63


Preview of Doc2.txt:
------------------------------------------------------------
[[PAGE 1]]
Guidelines for the management and
     care of diabetes mellitus





           Quick reference guide


Diabetes mellitus is one of the most common noncommunicable diseases worldwide,
and its epidemic proportion has placed it at the forefront of public health challenges.
In the Eastern Mediterranean Region, there has been a rapid increase in the incidence
of diabetes mellitus, mainly of type 2. It is now the fourth leading cause of death in the
Region and an estimated 22 million people have diabetes out of a total adult population
of 290 million. This quick reference guide aims to 


## Step 5: Load the Evaluation Questions
To measure which chunk setting retrieves best, we need questions with known answers (a "ground truth").

Each question in `questions.json` has:
- `question`: what a user would ask
- `doc`: the document that contains the answer
- `answer_phrase`: a short exact phrase from the raw text that must appear in a correct chunk

**How we define a correct chunk:** a chunk is *relevant* (a "gold chunk") if it comes from the right document and contains the answer phrase. Because chunk IDs change with every chunk size, we store the *phrase*, not a chunk ID, and find the gold chunks separately for each setting.

Before using the questions, we verify that every phrase really exists in our raw text.

In [6]:
# Load the evaluation questions
questions = json.loads(Path("questions.json").read_text(encoding="utf-8"))
questions_df = pd.DataFrame(questions)

# Verify that every answer phrase really exists in its document's raw text
def phrase_count(row):
    raw = (RAW_TEXT_DIR / f"{Path(row['doc']).stem}.txt").read_text(encoding="utf-8")
    return raw.count(row["answer_phrase"])

questions_df["found_in_text"] = questions_df.apply(phrase_count, axis=1)

print(f"Total questions: {len(questions_df)}")
print(f"Questions whose phrase was NOT found: {(questions_df['found_in_text'] == 0).sum()}")

display(questions_df[["qid", "doc", "question", "found_in_text"]])

Total questions: 16
Questions whose phrase was NOT found: 0


,qid,doc,question,found_in_text
0,Q01,Doc1.pdf,What is the WHO target blood pressure goal for...,2
1,Q02,Doc2.pdf,How is gestational diabetes defined?,1
2,Q03,Doc2.pdf,What fasting plasma glucose value indicates di...,1
3,Q04,Doc2.pdf,What HbA1c level is the target for good diabet...,1
4,Q05,Doc3.pdf,Which drug should pregnant women with uncompli...,2
5,Q06,Doc3.pdf,What single low dose of primaquine is given wi...,2
6,Q07,Doc4.pdf,When should ART initiation be offered after a ...,1
7,Q08,Doc5.pdf,Which antiviral prophylaxis is recommended for...,1
8,Q09,Doc6.pdf,Which interventions are recommended for adults...,1
9,Q10,Doc7.pdf,How are cholera patients with some dehydration...,1


## Step 6: Fixed-Size Chunking (12 settings)
We split every document into chunks of the **same length** by sliding a window across the text. We test 12 settings: 4 chunk sizes x 3 overlaps (0%, 10%, 20% of the size).

Each setting is named `S<size>_O<overlap>`. For example, `S1000_O200` means chunk size 1000 characters with an overlap of 200 characters.

| Chunk size | Overlap 0% | Overlap 10% | Overlap 20% |
|---|---|---|---|
| 256 | 0 | 26 | 51 |
| 500 | 0 | 50 | 100 |
| 1000 | 0 | 100 | 200 |
| 2000 | 0 | 200 | 400 |

**How it works:** the window moves forward by `size - overlap` characters each time. Overlap means the end of one chunk is repeated at the start of the next, so a sentence cut at a boundary keeps some context.

**Example:** with size 1000 and overlap 200, chunk 1 covers characters 0-1000, chunk 2 covers 800-1800, and chunk 3 covers 1600-2600.

For every chunk we store the document name, the position in the text and the page range, so each answer can be traced back to its source page.

**Known limitation:** because the size is fixed, a chunk can start or end in the middle of a word or sentence. This is expected for this method.

In this step we only build the chunks and count them. Search and scoring come later.

In [7]:
import bisect

# 12 settings: (label, chunk_size, overlap). Overlap is 0%, 10% and 20% of the size.
# Label format: S<size>_O<overlap>
CONFIGS = []
for size in [256, 500, 1000, 2000]:
    for pct in [0, 10, 20]:
        overlap = round(size * pct / 100)
        CONFIGS.append((f"S{size}_O{overlap}", size, overlap))

PAGE_MARKER = re.compile(r"\[\[PAGE (\d+)\]\]\n")


def load_pages(txt_path):
    """Read a raw .txt file and return a list of (page_num, page_text)."""
    content = txt_path.read_text(encoding="utf-8")
    parts = PAGE_MARKER.split(content)
    # parts = [before_first_marker, page_num, text, page_num, text, ...]
    return [(int(parts[i]), parts[i + 1]) for i in range(1, len(parts), 2)]


def build_full_text(pages):
    """Join all pages into one string and remember where each page starts."""
    full_text, page_starts, page_numbers = "", [], []
    for page_num, text in pages:
        page_starts.append(len(full_text))
        page_numbers.append(page_num)
        full_text += text + "\n"
    return full_text, page_starts, page_numbers


def chunk_text(full_text, size, overlap):
    """Return (start, end) character offsets of fixed-size overlapping chunks."""
    step = size - overlap
    spans = []
    for start in range(0, len(full_text), step):
        end = min(start + size, len(full_text))
        if full_text[start:end].strip():  # skip chunks with only whitespace
            spans.append((start, end))
        if end == len(full_text):
            break  # last chunk reached, avoid a tiny duplicate tail
    return spans


# Load every document once from the saved raw text files
RAW_TEXT_DIR = Path("raw_text")
docs = {}  # doc name -> (full_text, page_starts, page_numbers)
for txt_path in sorted(RAW_TEXT_DIR.glob("*.txt")):
    docs[txt_path.stem] = build_full_text(load_pages(txt_path))

total_original_chars = sum(len(d[0]) for d in docs.values())
print(f"Documents loaded: {len(docs)}")
print(f"Original text: {total_original_chars:,} characters\n")

# Build the chunks for every setting
chunk_store = {}  # label -> list of chunk dicts
for label, size, overlap in tqdm(CONFIGS, desc="Chunking"):
    chunks = []
    for doc_name, (text, page_starts, page_numbers) in docs.items():
        for k, (start, end) in enumerate(chunk_text(text, size, overlap)):
            p_start = bisect.bisect_right(page_starts, start) - 1
            p_end = bisect.bisect_right(page_starts, end - 1) - 1
            chunks.append(
                {
                    "chunk_id": f"{doc_name}_{k}",
                    "doc": doc_name,
                    "start": start,
                    "end": end,
                    "page_start": page_numbers[max(p_start, 0)],
                    "page_end": page_numbers[max(p_end, 0)],
                    "text": text[start:end],
                }
            )
    chunk_store[label] = chunks

# Summary table: number of chunks and average chunk length per setting
rows = []
for label, size, overlap in CONFIGS:
    chunks = chunk_store[label]
    rows.append(
        {
            "setting": label,
            "size": size,
            "overlap": overlap,
            "total_chunks": len(chunks),
            "avg_chars": round(sum(len(c["text"]) for c in chunks) / len(chunks), 1),
        }
    )
print("Summary of all 12 settings:")
display(pd.DataFrame(rows).set_index("setting"))

# Show two consecutive chunks of one setting so the overlap is visible
SHOW = "S500_O100"
sample = [c for c in chunk_store[SHOW] if c["doc"] == "Doc2"][3:5]
for c in sample:
    print("=" * 70)
    print(f"{c['chunk_id']} | chars {c['start']}-{c['end']} | pages {c['page_start']}-{c['page_end']}")
    print(c["text"])

Documents loaded: 10
Original text: 5,615,821 characters



Chunking:   0%|          | 0/12 [00:00<?, ?it/s]

Summary of all 12 settings:


,size,overlap,total_chunks,avg_chars
setting,,,,
S256_O0,256,0,21897,255.9
S256_O26,256,26,24372,255.9
S256_O51,256,51,27339,256.0
S500_O0,500,0,11230,499.8
S500_O50,500,50,12477,499.8
S500_O100,500,100,14038,499.9
S1000_O0,1000,0,5622,998.9
S1000_O100,1000,100,6244,999.2
S1000_O200,1000,200,7024,999.2


Doc2_3 | chars 1200-1700 | pages 2-2
should never be made on the
   basis of a single abnormal glucose value. Verification of the diagnosis with repeat testing
    is required, unless an individual presents with unequivocal hyperglycaemia, along with its
   classic symptoms.

•  The following table outlines the diagnostic values for diabetes mellitus and other categories
   of hyperglycaemia.





             Diagnostic values for diabetes mellitus and other categories
                                  of hyperglycaemia



       
Doc2_4 | chars 1600-2100 | pages 2-2
iabetes mellitus and other categories
                                  of hyperglycaemia



                                     Venous plasma glucose concentration
                                     mmol/L             mg/dL


   Diabetes mellitus
      fasting or                                  ≥7.0                 ≥126
     2-hour post-75 g glucose load               ≥11.1                ≥200
   IGT
      fasting (i

## Step 6b: Save the Chunks to .txt Files
We save the chunks of every setting to disk so we can read them directly and share them.

- One `.txt` file per setting, stored in the `chunks` folder (for example `S1000_O200.txt`)
- Every chunk starts with a header line: `===== CHUNK <id> | chars <start>-<end> | pages <first>-<last> =====`
- The header tells us the document, the chunk number, the position in the document text and the source pages
- Chunks are separated by a blank line

The chunk text is saved exactly as it was cut, including any word or sentence that was split in the middle.

In [8]:
# Folder where the chunk files will be saved
CHUNKS_DIR = Path("chunks")
CHUNKS_DIR.mkdir(exist_ok=True)

for label, size, overlap in CONFIGS:
    out_path = CHUNKS_DIR / f"{label}.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        for c in chunk_store[label]:
            f.write(
                f"===== CHUNK {c['chunk_id']} | chars {c['start']}-{c['end']} "
                f"| pages {c['page_start']}-{c['page_end']} =====\n"
            )
            f.write(c["text"])
            f.write("\n\n")

# Verify what was saved
saved = sorted(CHUNKS_DIR.glob("*.txt"))
print(f"Saved {len(saved)} files to: {CHUNKS_DIR.resolve()}\n")

check_df = pd.DataFrame(
    {
        "file": [p.name for p in saved],
        "chunks_in_file": [
            p.read_text(encoding="utf-8").count("===== CHUNK ") for p in saved
        ],
        "size_mb": [round(p.stat().st_size / (1024 * 1024), 2) for p in saved],
    }
)
display(check_df)

# Preview the start of one saved file
print("Preview of S500_O100.txt:")
print("-" * 60)
print((CHUNKS_DIR / "S500_O100.txt").read_text(encoding="utf-8")[:700])

Saved 12 files to: /home/as/Documents/week8/Med_Fixed_chunking/chunks



,file,chunks_in_file,size_mb
0,S1000_O0.txt,5622,5.73
1,S1000_O100.txt,6244,6.36
2,S1000_O200.txt,7024,7.16
3,S2000_O0.txt,2814,5.55
4,S2000_O200.txt,3123,6.16
5,S2000_O400.txt,3513,6.93
6,S256_O0.txt,21897,6.76
7,S256_O26.txt,24372,7.52
8,S256_O51.txt,27339,8.44
9,S500_O0.txt,11230,6.08


Preview of S500_O100.txt:
------------------------------------------------------------
===== CHUNK Doc1_0 | chars 0-500 | pages 1-4 =====
Guideline
for the
pharmacological
treatment of
hypertension
in adults





Guideline
for the
pharmacological
treatment of
hypertension
in adults


Guideline for the pharmacological treatment of hypertension in adults

ISBN 978-92-4-003398-6 (electronic version)
ISBN 978-92-4-003397-9 (print version)


© World Health Organization 2021

Some rights reserved. This work is available under the Creative Commons Attribution-NonCommercial-
ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO;
https://creativ

===== CHUNK Doc1_1 | chars 400-900 | pages 4-4 =====
 Commons Attribution-NonCommercial-
ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO;
https://c


In [9]:
# Find the correct chunks: right document AND contains the full answer phrase
correct = {}  # label -> {qid: [correct chunk ids]}
for label, size, overlap in CONFIGS:
    by_doc = {}
    for c in chunk_store[label]:
        by_doc.setdefault(c["doc"], []).append(c)

    correct[label] = {}
    for _, q in questions_df.iterrows():
        doc = Path(q["doc"]).stem
        phrase = q["answer_phrase"]
        correct[label][q["qid"]] = [c["chunk_id"] for c in by_doc[doc] if phrase in c["text"]]

# How many questions lost their answer in each setting
summary = pd.DataFrame(
    {
        "setting": [label for label, _, _ in CONFIGS],
        "total_chunks": [len(chunk_store[label]) for label, _, _ in CONFIGS],
        "questions_with_no_correct_chunk": [
            sum(len(ids) == 0 for ids in correct[label].values()) for label, _, _ in CONFIGS
        ],
    }
).set_index("setting")
display(summary)

Path("correct_chunks.json").write_text(json.dumps(correct, indent=2), encoding="utf-8")
print("Saved: correct_chunks.json")

,total_chunks,questions_with_no_correct_chunk
setting,,
S256_O0,21897,3
S256_O26,24372,3
S256_O51,27339,1
S500_O0,11230,4
S500_O50,12477,1
S500_O100,14038,0
S1000_O0,5622,3
S1000_O100,6244,0
S1000_O200,7024,0


Saved: correct_chunks.json


In [19]:
# Verify the FIRST and LAST chunking settings directly

for label in ["S256_O0", "S2000_O400"]:
    print("\n" + "=" * 80)
    print(f"SETTING: {label}")
    print("=" * 80)

    for _, q in questions_df.iterrows():
        qid = q["qid"]
        phrase = q["answer_phrase"]
        correct_ids = correct[label][qid]

        print(f"\n{qid}")
        print(f"Answer phrase: {phrase}")
        print(f"Correct chunk IDs: {correct_ids}")

        if len(correct_ids) == 0:
            print("❌ NO correct chunk contains the full answer phrase")
        else:
            print(f"✅ {len(correct_ids)} correct chunk(s) contain the full answer phrase")


SETTING: S256_O0

Q01
Answer phrase: target blood pressure treatment goal of <140/90 mmHg
Correct chunk IDs: ['Doc1_76', 'Doc1_309']
✅ 2 correct chunk(s) contain the full answer phrase

Q02
Answer phrase: Gestational diabetes is a state of carbohydrate intolerance
Correct chunk IDs: ['Doc2_13']
✅ 1 correct chunk(s) contain the full answer phrase

Q03
Answer phrase: ≥7.0 mmol/L (≥126 mg/dL)
Correct chunk IDs: ['Doc2_21']
✅ 1 correct chunk(s) contain the full answer phrase

Q04
Answer phrase: HbA1c (%)                          <6.0            <7.0
Correct chunk IDs: []
❌ NO correct chunk contains the full answer phrase

Q05
Answer phrase: should be treated with artemether-lumefantrine
Correct chunk IDs: ['Doc3_3687']
✅ 1 correct chunk(s) contain the full answer phrase

Q06
Answer phrase: a single dose of 0.25 mg/kg bw primaquine should be given with an ACT
Correct chunk IDs: ['Doc3_3615']
✅ 1 correct chunk(s) contain the full answer phrase

Q07
Answer phrase: Rapid ART initiation should

## Step 7: Find the Best Setting (Recall@5, Precision@5, MRR)
We test which chunk setting helps search find the right text. We use the 5 settings in which no question lost its answer in Step 7.

**How the test works** (for each setting):
1. Every chunk and every question is converted into numbers (an *embedding*) with the model `BAAI/bge-small-en-v1.5`.
2. For each question, we take the **5 most similar chunks** (top 5).
3. We compare these 5 chunks with the question's **correct chunks** from Step 7.

**Metrics** (computed per question, then averaged over the 16 questions):
- **Recall@5** = correct chunks found in the top 5 / total correct chunks of that question
- **Precision@5** = correct chunks found in the top 5 / 5
- **MRR** = 1 / rank of the first correct chunk (rank 1 gives 1.0, rank 2 gives 0.5, not in the top 5 gives 0)

**How we pick the best setting:** highest MRR first, then highest Recall@5. Precision@5 and fewer chunks break ties.

**Limits:**
- Only 16 questions, so very small differences (for example 0.02 in MRR) may be luck.
- The model reads about 512 tokens, so very long chunks (2000 characters) may be cut at the end.
- Embeddings are saved in the `embeddings` folder, so a re-run is fast.

In [10]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer

TOP_K = 5
MODEL_NAME = "BAAI/bge-small-en-v1.5"
# bge models work better when the question starts with this instruction
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

# Settings to test. To test all 12, use: RETRIEVAL_LABELS = [c[0] for c in CONFIGS]
RETRIEVAL_LABELS = ["S500_O100", "S1000_O100", "S1000_O200", "S2000_O200", "S2000_O400"]

model = SentenceTransformer(MODEL_NAME, device="cpu")
model.max_seq_length = 512

EMB_DIR = Path("embeddings")
EMB_DIR.mkdir(exist_ok=True)

# Embed the 16 questions once
query_emb = model.encode(
    [QUERY_PREFIX + q for q in questions_df["question"]],
    normalize_embeddings=True,
)

rows = []
for label in RETRIEVAL_LABELS:
    chunks = chunk_store[label]
    chunk_ids = [c["chunk_id"] for c in chunks]

    # Embed all chunks of this setting (saved to disk, so re-runs are fast)
    emb_path = EMB_DIR / f"{label}.npy"
    if emb_path.exists():
        chunk_emb = np.load(emb_path)
    else:
        print(f"Embedding {len(chunks):,} chunks for {label} ...")
        chunk_emb = model.encode(
            [c["text"] for c in chunks],
            batch_size=64,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        np.save(emb_path, chunk_emb)

    # Top-5 most similar chunks for every question (cosine similarity)
    scores = query_emb @ chunk_emb.T
    top_idx = np.argsort(-scores, axis=1)[:, :TOP_K]

    recalls, precisions, rrs = [], [], []
    for i, qid in enumerate(questions_df["qid"]):
        retrieved = [chunk_ids[j] for j in top_idx[i]]
        correct_ids = set(correct[label][qid])
        found = [r in correct_ids for r in retrieved]
        n_found = sum(found)

        recalls.append(n_found / len(correct_ids) if correct_ids else 0.0)
        precisions.append(n_found / TOP_K)
        rrs.append(1 / (found.index(True) + 1) if n_found else 0.0)

    rows.append(
        {
            "setting": label,
            "total_chunks": len(chunks),
            "recall@5": round(float(np.mean(recalls)), 3),
            "precision@5": round(float(np.mean(precisions)), 3),
            "MRR": round(float(np.mean(rrs)), 3),
        }
    )

# Sort: MRR first, then Recall, then Precision, then fewer chunks
results_df = (
    pd.DataFrame(rows)
    .sort_values(
        ["MRR", "recall@5", "precision@5", "total_chunks"],
        ascending=[False, False, False, True],
    )
    .reset_index(drop=True)
)
print("\nResults, best first:")
display(results_df)

best = results_df.iloc[0]
print(
    f"BEST SETTING: {best['setting']}  "
    f"(MRR={best['MRR']}, Recall@5={best['recall@5']}, Precision@5={best['precision@5']})"
)

results_df.to_csv("retrieval_results.csv", index=False)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding 3,123 chunks for S2000_O200 ...


Batches:   0%|          | 0/49 [00:00<?, ?it/s]

Embedding 3,513 chunks for S2000_O400 ...


Batches:   0%|          | 0/55 [00:00<?, ?it/s]


Results, best first:


,setting,total_chunks,recall@5,precision@5,MRR
0,S2000_O400,3513,0.812,0.238,0.747
1,S2000_O200,3123,0.844,0.213,0.609
2,S500_O100,14038,0.656,0.163,0.575
3,S1000_O200,7024,0.812,0.225,0.574
4,S1000_O100,6244,0.719,0.163,0.573


BEST SETTING: S2000_O400  (MRR=0.747, Recall@5=0.812, Precision@5=0.238)


## Step 8: Inspect Two Questions in the Top 2 Settings
We take the two best settings from Step 8 and look closely at two questions, to see:
- which chunks the search returned (top 5), and where they came from
- which of them are **correct** (they contain the full answer phrase)
- how **Recall@5, Precision@5 and the reciprocal rank (RR)** are calculated for that question

**Reading the calculation:**
- Recall@5 = correct chunks found in the top 5 / total correct chunks of the question
- Precision@5 = correct chunks found in the top 5 / 5
- RR = 1 / rank of the first correct chunk (0 if none in the top 5)

MRR is the average of RR over all questions. Here we only show 2 questions, so the MRR shown is just the average of those 2 values (the real MRR in Step 8 uses all 16).

In [18]:
import re
import textwrap
import numpy as np
import pandas as pd

# ======================= CONFIG =======================
QIDS = ["Q01","Q02","Q03","Q04","Q05","Q06","Q07","Q08","Q09","Q10","Q11","Q12","Q13","Q14","Q15" ,"Q16"]      # questions to inspect (change freely)
PREVIEW_CHARS = 450        # how much of each chunk to print
WIDTH = 100                # output width
# ======================================================


def norm(s):
    """Collapse all whitespace so text is easier to read."""
    return re.sub(r"\s+", " ", s).strip()


def wrap(text, indent=8):
    return textwrap.fill(text, width=WIDTH, initial_indent=" " * indent,
                         subsequent_indent=" " * indent)


def snippet(text, phrase, n):
    """Show the chunk text; if the answer phrase is inside, mark it with [[ ]]."""
    t, p = norm(text), norm(phrase)
    i = t.lower().find(p.lower())
    if i == -1:
        return t[:n] + (" ..." if len(t) > n else "")
    j = i + len(p)
    extra = max(0, n - len(p))
    start = max(0, i - extra // 2)
    end = min(len(t), j + extra - (i - start))
    return (("... " if start > 0 else "") + t[start:i] + "[[" + t[i:j] + "]]"
            + t[j:end] + (" ..." if end < len(t) else ""))


top2 = results_df.head(2)["setting"].tolist()   # best 2 settings from the retrieval test
emb = {lb: np.load(EMB_DIR / f"{lb}.npy") for lb in top2}
rows = []   # one record per (setting, question)

for qid in QIDS:
    m = questions_df.index[questions_df["qid"] == qid]
    if len(m) == 0:
        print(f"\n{qid} not found in questions_df, skipping.")
        continue
    qi = m[0]
    question = questions_df.loc[qi, "question"]
    phrase = questions_df.loc[qi, "answer_phrase"]
    q_vec = model.encode([QUERY_PREFIX + question], normalize_embeddings=True)[0]

    print("\n" + "#" * WIDTH)
    print(f"QUESTION {qid}: {question}")
    print(f"ANSWER PHRASE: '{phrase}'")
    print("#" * WIDTH)

    for label in top2:
        chunks = chunk_store[label]
        chunk_ids = [x["chunk_id"] for x in chunks]
        by_id = {x["chunk_id"]: x for x in chunks}
        scores = emb[label] @ q_vec
        top_idx = np.argsort(-scores)[:TOP_K]
        correct_ids = set(correct[label][qid])

        flags = [chunk_ids[j] in correct_ids for j in top_idx]
        n_found, n_correct = sum(flags), len(correct_ids)
        first_rank = flags.index(True) + 1 if n_found else None
        found_txt = f"FOUND at rank {first_rank}" if first_rank else f"NOT FOUND in top {TOP_K}"
        cut_txt = "NO" if n_correct else "YES (no chunk holds the full phrase)"

        print("\n" + "=" * WIDTH)
        print(f">>> {label} | {len(chunks):,} chunks | {found_txt} | answer cut by chunking: {cut_txt}")
        print(f"    All correct chunks for this question ({n_correct}): {sorted(correct_ids)}")
        print("=" * WIDTH)

        for rank, j in enumerate(top_idx, start=1):
            cid = chunk_ids[j]
            ch = by_id[cid]
            mark = "CORRECT" if flags[rank - 1] else "-"
            print(f"\n    Rank {rank} | {mark} | {cid} | {len(ch['text'])} chars | "
                  f"pages {ch['page_start']}-{ch['page_end']} | similarity={scores[j]:.2f}")
            print(wrap(snippet(ch["text"], phrase, PREVIEW_CHARS)))

        # Metrics for this question in this setting
        recall = n_found / n_correct if n_correct else 0.0
        precision = n_found / TOP_K
        rr = 1 / first_rank if first_rank else 0.0

        print(f"\n    Metrics ({qid}, {label})")
        print(f"      Recall@{TOP_K}    = {n_found} / {n_correct} = {recall:.3f}")
        print(f"      Precision@{TOP_K} = {n_found} / {TOP_K} = {precision:.3f}")
        print("      MRR           = " + (f"1 / {first_rank} = {rr:.3f}" if first_rank else "0 (no correct chunk in top 5)"))

        rows.append({"qid": qid, "setting": label, "chunks": len(chunks),
                     "found_at_rank": first_rank if first_rank else "-",
                     "correct_found": n_found, "correct_total": n_correct,
                     f"Recall@{TOP_K}": round(recall, 3),
                     f"Precision@{TOP_K}": round(precision, 3),
                     "MRR": round(rr, 3)})

# ======================= SUMMARY =======================
if rows:
    df = pd.DataFrame(rows)
    print("\n" + "#" * WIDTH)
    print(f"SUMMARY PER QUESTION (TOP_K = {TOP_K})")
    print("#" * WIDTH)
    print(df.to_string(index=False))

    agg = (df.groupby("setting", sort=False)
             .agg(chunks=("chunks", "first"),
                  questions=("qid", "count"),
                  **{f"Recall@{TOP_K}": (f"Recall@{TOP_K}", "mean"),
                     f"Precision@{TOP_K}": (f"Precision@{TOP_K}", "mean"),
                     "MRR": ("MRR", "mean")})
             .round(3).reset_index())
    print("\n" + "#" * WIDTH)
    print(f"SUMMARY PER SETTING (average over the {len(QIDS)} inspected questions only)")
    print("#" * WIDTH)
    print(agg.to_string(index=False))


####################################################################################################
QUESTION Q01: What is the WHO target blood pressure goal for adults with hypertension without comorbidities?
ANSWER PHRASE: 'target blood pressure treatment goal of <140/90 mmHg'
####################################################################################################

>>> S2000_O400 | 3,513 chunks | FOUND at rank 1 | answer cut by chunking: NO
    All correct chunks for this question (3): ['Doc1_12', 'Doc1_50', 'Doc1_51']

    Rank 1 | CORRECT | Doc1_51 | 2000 chars | pages 27-28 | similarity=0.82
        15 3.6 Target blood pressure 6. RECOMMENDATION ON TARGET BLOOD PRESSURES WHO recommends a
        [[target blood pressure treatment goal of <140/90 mmHg]] in all patients with hypertension
        without comorbidities. Strong recommendation, moderate-certainty evidence WHO recommends a
        target systolic blood pressure treatment goal of <130 mmHg in patients with hyp

In [29]:
from pathlib import Path
import re

CHUNK_DIR = Path("chunks")

answer_phrase = "target blood pressure treatment goal of <140/90 mmHg"

HEADER_PATTERN = re.compile(
    r"===== CHUNK (.*?) \| chars .*? \| pages .*? ====="
)

for file_path in [
    CHUNK_DIR / "S2000_O400.txt",
    CHUNK_DIR / "S2000_O200.txt"
]:

    print("\n" + "=" * 100)
    print(f"SETTING: {file_path.stem}")
    print("=" * 100)

    text = file_path.read_text(encoding="utf-8")

    headers = list(HEADER_PATTERN.finditer(text))

    matching_chunks = []

    for i, header in enumerate(headers):

        chunk_id = header.group(1)

        # Current chunk ka text next header tak
        if i + 1 < len(headers):
            chunk_text = text[header.start():headers[i + 1].start()]
        else:
            chunk_text = text[header.start():]

        # Answer phrase automatically search karo
        if answer_phrase.lower() in chunk_text.lower():
            matching_chunks.append(chunk_id)

            print(f"\nFOUND: {chunk_id}")
            print(chunk_text.strip())

    print("\n" + "-" * 100)
    print(f"Total matching chunks: {len(matching_chunks)}")
    print(f"Chunk IDs: {matching_chunks}")


SETTING: S2000_O400

FOUND: Doc1_12
===== CHUNK Doc1_12 | chars 19200-21200 | pages 10-11 =====
nce


            5. RECOMMENDATION ON COMBINATION THERAPY

          For adults with hypertension requiring pharmacological treatment, WHO suggests
         combination therapy, preferably with a single-pill combination (to improve adherence and
           persistence), as an initial treatment. Antihypertensive medications used in combination
         therapy should be chosen from the following three drug classes: diuretics (thiazide or
           thiazide-like), angiotensin-converting enzyme inhibitors (ACEis)/angiotensin-receptor
          blockers (ARBs), and long-acting dihydropyridine calcium channel blockers (CCBs).

          Conditional recommendation, moderate-certainty evidence


            6. RECOMMENDATIONS ON TARGET BLOOD PRESSURE

      WHO recommends a target blood pressure treatment goal of <140/90 mmHg in all patients
         with hypertension without comorbidities.

   